In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
%matplotlib inline

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

sns.set_theme(style="whitegrid")

In [9]:
df = pd.read_csv("../data/raw/online_retail_II.csv")

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


In [11]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Null counts and percentages per column

In [12]:
null_counts = df.isnull().sum()
null_percentages = (df.isnull().sum() / len(df)) * 100

# Combine into a clean summary DataFrame
null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Null Percentage (%)': null_percentages
}).sort_values(by='Null Percentage (%)', ascending=False)

# Display the result
display(null_summary)

,Null Count,Null Percentage (%)
Customer ID,243007,22.766873
Description,4382,0.410541
Invoice,0,0.000000
StockCode,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
Price,0,0.000000
Country,0,0.000000


~22% of `CustomerID`s are missing. This means we have ~22% people checking out as guests.

Now we check for cancellations and non-product line items.

In [14]:
# Sample rows where 'Invoice' starts with "C" (Cancellations)
cancellations = df[df['Invoice'].astype(str).str.startswith('C', na=False)]
print(f"Number of cancellation rows: {len(cancellations):,}")
print("Sample of cancellations:")
display(cancellations.head())

Number of cancellation rows: 19,494
Sample of cancellations:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [16]:
# Unique values in 'StockCode' that appear to be non-product codes
# Using a regular expression to find codes that are entirely alphabetical (no numbers)
non_product_mask = df['StockCode'].astype(str).str.contains('^[A-Za-z\s]+$', regex=True, na=False)
non_product_codes = df.loc[non_product_mask, 'StockCode'].value_counts()

print("\nUnique non-product StockCodes found:")
display(non_product_codes)

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
C:\Users\BHAVY\AppData\Local\Temp\ipykernel_34856\771542194.py:3: SyntaxWarning: invalid escape sequence '\s'
  non_product_mask = df['StockCode'].astype(str).str.contains('^[A-Za-z\s]+$', regex=True, na=False)



Unique non-product StockCodes found:


StockCode
POST            2122
DOT             1446
M               1421
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
CRUK              16
B                  6
m                  5
DCGSLGIRL          1
DCGSLBOY           1
GIFT               1
Name: count, dtype: int64

We've identified the cancellations and non-product line items and their counts.

Now we look at the distribution of price and order quantity within the dataset.

In [17]:
# Calculating distribution metrics for Price
price_min = df['Price'].min()
price_max = df['Price'].max()
price_median = df['Price'].median()
price_zero_or_less_pct = (df['Price'] <= 0).mean() * 100

print("--- Price Distribution ---")
print(f"Minimum Price: £{price_min:.2f}")
print(f"Maximum Price: £{price_max:.2f}")
print(f"Median Price:  £{price_median:.2f}")
print(f"Rows with Price <= 0: {price_zero_or_less_pct:.2f}%")

--- Price Distribution ---
Minimum Price: £-53594.36
Maximum Price: £38970.00
Median Price:  £2.10
Rows with Price <= 0: 0.58%


In [18]:
# Calculate distribution metrics for Quantity
qty_min = df['Quantity'].min()
qty_max = df['Quantity'].max()
qty_median = df['Quantity'].median()
qty_zero_or_less_pct = (df['Quantity'] <= 0).mean() * 100

print("\n--- Quantity Distribution ---")
print(f"Minimum Quantity: {qty_min:,}")
print(f"Maximum Quantity: {qty_max:,}")
print(f"Median Quantity:  {qty_median:,}")
print(f"Rows with Quantity <= 0: {qty_zero_or_less_pct:.2f}%")


--- Quantity Distribution ---
Minimum Quantity: -80,995
Maximum Quantity: 80,995
Median Quantity:  3.0
Rows with Quantity <= 0: 2.15%


What stands out from this analysis:
<ul><li> <b>The +/- 80,995 symmetry:</b> Max quantity is 80,995 and the min is exactly -80,995. That probably represents an erroneous order that was immediately reversed/cancelled.

<li> <b>Negative Prices:</b> A price of £-53594.36 is completely illogical for a retail purchase. This is likely a mistake.

<li><b>The Percentages:</b> Rows with `Price <= 0` make up <1% of the data, and `Quantity <= 0` make up ~2.15%.</ul>

Now, we analyse the positive price and order quantity. 

In [22]:
# Filter for strictly positive, "commercial" data
positive_data = df[(df['Price'] > 0) & (df['Quantity'] > 0)]

# Define percentiles to see the spread, specifically looking at the 95th and 99th
percentiles = [0.25, 0.5, 0.75, 0.95, 0.99]

print("--- Distribution of Positive Prices ---")
display(positive_data['Price'].describe(percentiles=percentiles).to_frame().T)

print("\n--- Distribution of Positive Quantities ---")
display(positive_data['Quantity'].describe(percentiles=percentiles).to_frame().T)

--- Distribution of Positive Prices ---


,count,mean,std,min,25%,50%,75%,95%,99%,max
Price,1041671.0,4.077038,51.448979,0.001,1.25,2.1,4.13,9.95,18.0,25111.09



--- Distribution of Positive Quantities ---


,count,mean,std,min,25%,50%,75%,95%,99%,max
Quantity,1041671.0,10.963448,126.51493,1.0,1.0,3.0,10.0,30.0,100.0,80995.0


Here is what these tables tell us:
<ol><li><b>It is a low-ticket, high-volume business:</b> Look at the 50% (median) and 99% columns for Price. Half of everything they sell costs £2.10 or less. 99% of all transactions involve items priced at £18.00 or less.

<li><b>The Max values are still hiding anomalies:</b> Even after filtering out negative prices and quantities, our max price is £25,111.09 and our max quantity is 80,995. Because we know 99% of our products cost under £18, the £25k item is likely another accounting code (like an AMAZONFEE or manual adjustment) that we'll filter out later.

<li><b>Wholesale vs. Retail behavior:</b> The quantities show that 75% of purchases are for 10 units or less, but the 99th percentile jumps to 100 units. This suggests a mix of retail and wholesale customers (as specified by the datacard).</ol>

<br>
Now we analyse the country distribution in the dataset:

In [35]:
print("--- Top 10 Countries by Transaction Count ---")
display(df['Country'].value_counts().head(10))
print("--- Unique Countires: Names + Counts ---")
print(f"Number of unique countries = {len(df["Country"].unique())}")
print("Names of unique countires:")
print(df["Country"].unique())

--- Top 10 Countries by Transaction Count ---


Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

--- Unique Countires: Names + Counts ---
Number of unique countries = 43
Names of unique countires:
['United Kingdom' 'France' 'USA' 'Belgium' 'Australia' 'EIRE' 'Germany'
 'Portugal' 'Japan' 'Denmark' 'Nigeria' 'Netherlands' 'Poland' 'Spain'
 'Channel Islands' 'Italy' 'Cyprus' 'Greece' 'Norway' 'Austria' 'Sweden'
 'United Arab Emirates' 'Finland' 'Switzerland' 'Unspecified' 'Malta'
 'Bahrain' 'RSA' 'Bermuda' 'Hong Kong' 'Singapore' 'Thailand' 'Israel'
 'Lithuania' 'West Indies' 'Lebanon' 'Korea' 'Brazil' 'Canada' 'Iceland'
 'Saudi Arabia' 'Czech Republic' 'European Community']


In [36]:
# Fill nulls with an empty string temporarily to make searching easier
anomalous_descriptions = df[df['Description'].fillna('').str.strip() == '?']

print(f"Number of rows where Description is '?': {len(anomalous_descriptions):,}")
print("Sample of '?' descriptions:")
display(anomalous_descriptions)

Number of rows where Description is '?': 92
Sample of '?' descriptions:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
63439,495027,21314,?,1146,2010-01-20 13:48:00,0.0,NaN,United Kingdom
100428,498887,20679,?,-330,2010-02-23 13:04:00,0.0,NaN,United Kingdom
167838,505285,85221,?,-1120,2010-04-21 11:48:00,0.0,NaN,United Kingdom
171218,505638,21929,?,-420,2010-04-23 13:41:00,0.0,NaN,United Kingdom
183682,506866,84270,?,-3000,2010-05-04 15:20:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
918136,570712,22121,?,-90,2011-10-12 10:15:00,0.0,NaN,United Kingdom
920632,571024,22812,?,-270,2011-10-13 12:18:00,0.0,NaN,United Kingdom
922487,571116,23131,?,-49,2011-10-13 17:46:00,0.0,NaN,United Kingdom
922686,571126,72802A,?,-50,2011-10-14 09:53:00,0.0,NaN,United Kingdom


What do these outputs tell us?
<ol>
<li><b>UK Dominance</b>: The UK accounts for 981,330 of the total 1,067,371 (~91.9%) of all transactions. We will focus only on UK data.
<li><b>The ? Descriptions</b>: the `Price` is exactly 0.0 for all of them, the `CustomerID` is missing (`NaN`), and the `Quantity` values are not within the Inter-Quartile Range (IQR). They are likely to be internal system adjustments such as warehouse staff writing off lost or damaged inventory.
</ol>

<b> Here is a summary of the findings in this notebook</b> 
<ol>
<li> <b>Shape & Missing Data:</b> The dataset contains ~1.06 million rows. `CustomerID` is missing in ~22% of rows, and `Description` is missing in ~0.4% of rows.
<li> <b>Cancellations:</b> There are 19,494 rows where the `Invoice` starts with "C". These correspond to negative quantities (returns/cancellations) which we will later remove to accurately model baseline demand.
<li> <b>Non-Product Codes:</b> Multiple text-only `StockCode` values exist (e.g., `POST`, `DOT`, `M`, `AMAZONFEE`, `BANK CHARGES`). These are administrative fees or postage, not physical retail products, and must be excluded.
<li> <b>Pricing & Quantity Anomalies:</b>
    <ul>
    <li> ~0.58% of rows have a Price < 0. 
    <li> ~2.15% of rows have a Quantity < 0. 
    <li> Analysis of the 99th percentile shows standard retail behavior (99% of positive-priced items are <£18.00). Rows with zero/negative prices or quantities represent system errors or returns, etc and should be dropped.
    </ul>
<li> <b>Geographic Scope:</b> The dataset spans 43 unique countries, but the UK accounts for the majority. We will focus only on the UK.
<li> <b>System Errors:</b> Rows with a strictly `?` description (92 rows) consistently exhibit £0.00 prices and missing `CustomerID`s, indicating internal adjustments rather than real sales.